## importamos las librerias necesarias 

In [38]:
from pathlib import Path
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.types import Integer, String, Date, BigInteger, Numeric
import plotly.express as px
import plotly.graph_objects as go

## Extraemos la información

In [4]:
DATA_PATH = Path(r"C:\Users\sebas\Downloads\ncr_ride_bookings.csv")

In [5]:
def extract_data(data_path: Path) -> pd.DataFrame:
    df = pd.read_csv(data_path)

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    print("Dataset extraído correctamente")
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")

    return df

## Generamos las dimensiones 

In [6]:
def transform_data(df: pd.DataFrame) -> dict:
    df = df.copy()

    # =========================
    # Limpieza base sin perder datos
    # =========================
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["booking_id"] = (
        df["booking_id"]
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    df["customer_id"] = (
        df["customer_id"]
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    text_cols = [
        "booking_status",
        "vehicle_type",
        "pickup_location",
        "drop_location",
        "payment_method",
        "reason_for_cancelling_by_customer",
        "driver_cancellation_reason",
        "incomplete_rides_reason",
    ]

    for col in text_cols:
        df[col] = df[col].astype("string").str.strip()

    numeric_cols = [
        "avg_vtat",
        "avg_ctat",
        "cancelled_rides_by_customer",
        "cancelled_rides_by_driver",
        "incomplete_rides",
        "booking_value",
        "ride_distance",
        "driver_ratings",
        "customer_rating",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # =========================
    # 1. Dimensión Fecha
    # =========================
    dim_date = df[["date"]].drop_duplicates().copy()
    dim_date = dim_date.dropna(subset=["date"])
    dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
    dim_date["dia"] = dim_date["date"].dt.day
    dim_date["mes"] = dim_date["date"].dt.month
    dim_date["nombre_mes"] = dim_date["date"].dt.month_name()
    dim_date["trimestre"] = dim_date["date"].dt.quarter
    dim_date["anio"] = dim_date["date"].dt.year
    dim_date["dia_semana"] = dim_date["date"].dt.day_name()

    dim_date = dim_date[
        [
            "date_key",
            "date",
            "dia",
            "mes",
            "nombre_mes",
            "trimestre",
            "anio",
            "dia_semana",
        ]
    ].rename(columns={"date": "fecha"})

    # =========================
    # 2. Dimensión Cliente
    # =========================
    dim_customer = df[["customer_id"]].drop_duplicates().copy()
    dim_customer = dim_customer.sort_values("customer_id").reset_index(drop=True)
    dim_customer["customer_key"] = dim_customer.index + 1
    dim_customer = dim_customer[["customer_key", "customer_id"]]

    # =========================
    # 3. Dimensión Vehículo
    # =========================
    dim_vehicle = df[["vehicle_type"]].drop_duplicates().copy()
    dim_vehicle = dim_vehicle.sort_values("vehicle_type").reset_index(drop=True)
    dim_vehicle["vehicle_key"] = dim_vehicle.index + 1
    dim_vehicle = dim_vehicle[["vehicle_key", "vehicle_type"]]

    # =========================
    # 4. Dimensión Ubicación
    # =========================
    dim_location = df[["pickup_location", "drop_location"]].drop_duplicates().copy()
    dim_location = (
        dim_location
        .sort_values(["pickup_location", "drop_location"])
        .reset_index(drop=True)
    )
    dim_location["location_key"] = dim_location.index + 1
    dim_location = dim_location[
        ["location_key", "pickup_location", "drop_location"]
    ]

    # =========================
    # 5. Dimensión Método de Pago
    # =========================
    dim_payment = df[["payment_method"]].copy()
    dim_payment["payment_method"] = dim_payment["payment_method"].fillna("No aplica")
    dim_payment = dim_payment.drop_duplicates()
    dim_payment = dim_payment.sort_values("payment_method").reset_index(drop=True)
    dim_payment["payment_key"] = dim_payment.index + 1
    dim_payment = dim_payment[["payment_key", "payment_method"]]

    # =========================
    # 6. Dimensión Estado
    # =========================
    dim_status = df[
        [
            "booking_status",
            "reason_for_cancelling_by_customer",
            "driver_cancellation_reason",
            "incomplete_rides_reason",
        ]
    ].copy()

    dim_status = dim_status.rename(
        columns={
            "reason_for_cancelling_by_customer": "cancellation_reason_customer",
            "driver_cancellation_reason": "cancellation_reason_driver",
            "incomplete_rides_reason": "incomplete_reason",
        }
    )

    dim_status = dim_status.fillna("No aplica")
    dim_status = dim_status.drop_duplicates()

    dim_status = dim_status.sort_values(
        [
            "booking_status",
            "cancellation_reason_customer",
            "cancellation_reason_driver",
            "incomplete_reason",
        ]
    ).reset_index(drop=True)

    dim_status["status_key"] = dim_status.index + 1
    dim_status = dim_status[
        [
            "status_key",
            "booking_status",
            "cancellation_reason_customer",
            "cancellation_reason_driver",
            "incomplete_reason",
        ]
    ]

    # =========================
    # 7. Tabla de Hechos
    # =========================
    fact = df.copy()
    fact["date_key"] = fact["date"].dt.strftime("%Y%m%d").astype(int)
    fact["payment_method"] = fact["payment_method"].fillna("No aplica")

    fact = fact.merge(dim_customer, on="customer_id", how="left")
    fact = fact.merge(dim_vehicle, on="vehicle_type", how="left")
    fact = fact.merge(dim_location, on=["pickup_location", "drop_location"], how="left")
    fact = fact.merge(dim_payment, on="payment_method", how="left")

    fact_status_cols = [
        "booking_status",
        "reason_for_cancelling_by_customer",
        "driver_cancellation_reason",
        "incomplete_rides_reason",
    ]

    fact[fact_status_cols] = fact[fact_status_cols].fillna("No aplica")

    dim_status_merge = dim_status.rename(
        columns={
            "cancellation_reason_customer": "reason_for_cancelling_by_customer",
            "cancellation_reason_driver": "driver_cancellation_reason",
            "incomplete_reason": "incomplete_rides_reason",
        }
    )

    fact = fact.merge(
        dim_status_merge,
        on=fact_status_cols,
        how="left"
    )

    fact["booking_key"] = range(1, len(fact) + 1)

    fact["is_completed"] = (fact["booking_status"] == "Completed").astype(int)
    fact["is_cancelled_customer"] = (fact["booking_status"] == "Cancelled by Customer").astype(int)
    fact["is_cancelled_driver"] = (fact["booking_status"] == "Cancelled by Driver").astype(int)
    fact["is_no_driver_found"] = (fact["booking_status"] == "No Driver Found").astype(int)
    fact["is_incomplete"] = (fact["booking_status"] == "Incomplete").astype(int)

    fact_bookings = fact[
        [
            "booking_key",
            "booking_id",
            "date_key",
            "customer_key",
            "vehicle_key",
            "location_key",
            "payment_key",
            "status_key",
            "avg_vtat",
            "avg_ctat",
            "booking_value",
            "ride_distance",
            "driver_ratings",
            "customer_rating",
            "is_completed",
            "is_cancelled_customer",
            "is_cancelled_driver",
            "is_no_driver_found",
            "is_incomplete",
        ]
    ].copy()

    fact_bookings = fact_bookings.rename(
        columns={"driver_ratings": "driver_rating"}
    )

    return {
        "dim_date": dim_date,
        "dim_customer": dim_customer,
        "dim_vehicle": dim_vehicle,
        "dim_location": dim_location,
        "dim_payment": dim_payment,
        "dim_status": dim_status,
        "fact_bookings": fact_bookings,
    }

## limpieza sobre las dimensiones y fact (Transform)

In [7]:
def clean_dimensions_and_fact(tables: dict) -> dict:
    tables = tables.copy()

    # =========================
    # Limpieza dim_date
    # =========================
    tables["dim_date"]["fecha"] = pd.to_datetime(
        tables["dim_date"]["fecha"],
        errors="coerce"
    )

    tables["dim_date"] = tables["dim_date"].dropna(subset=["fecha"])
    tables["dim_date"]["date_key"] = tables["dim_date"]["date_key"].astype(int)

    # =========================
    # Limpieza dim_customer
    # =========================
    tables["dim_customer"]["customer_id"] = (
        tables["dim_customer"]["customer_id"]
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    # =========================
    # Limpieza dim_vehicle
    # =========================
    tables["dim_vehicle"]["vehicle_type"] = (
        tables["dim_vehicle"]["vehicle_type"]
        .astype(str)
        .str.strip()
    )

    # =========================
    # Limpieza dim_location
    # =========================
    for col in ["pickup_location", "drop_location"]:
        tables["dim_location"][col] = (
            tables["dim_location"][col]
            .astype(str)
            .str.strip()
        )

    # =========================
    # Limpieza dim_payment
    # =========================
    tables["dim_payment"]["payment_method"] = (
        tables["dim_payment"]["payment_method"]
        .fillna("No aplica")
        .astype(str)
        .str.strip()
    )

    # =========================
    # Limpieza dim_status
    # =========================
    status_cols = [
        "booking_status",
        "cancellation_reason_customer",
        "cancellation_reason_driver",
        "incomplete_reason",
    ]

    for col in status_cols:
        tables["dim_status"][col] = (
            tables["dim_status"][col]
            .fillna("No aplica")
            .astype(str)
            .str.strip()
        )

    # =========================
    # Limpieza fact_bookings
    # =========================
    tables["fact_bookings"]["booking_id"] = (
        tables["fact_bookings"]["booking_id"]
        .astype(str)
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    numeric_cols = [
        "avg_vtat",
        "avg_ctat",
        "booking_value",
        "ride_distance",
        "driver_rating",
        "customer_rating",
    ]

    for col in numeric_cols:
        tables["fact_bookings"][col] = pd.to_numeric(
            tables["fact_bookings"][col],
            errors="coerce"
        )

    key_cols = [
        "booking_key",
        "date_key",
        "customer_key",
        "vehicle_key",
        "location_key",
        "payment_key",
        "status_key",
    ]

    for col in key_cols:
        tables["fact_bookings"][col] = tables["fact_bookings"][col].astype(int)

    flag_cols = [
        "is_completed",
        "is_cancelled_customer",
        "is_cancelled_driver",
        "is_no_driver_found",
        "is_incomplete",
    ]

    for col in flag_cols:
        tables["fact_bookings"][col] = (
            tables["fact_bookings"][col]
            .fillna(0)
            .astype(int)
        )

    return tables

## validación que todo esta bien

In [9]:
df = extract_data(DATA_PATH)

tables = transform_data(df)

tables = clean_dimensions_and_fact(tables)

validate_tables(tables, df)

Dataset extraído correctamente
Filas: 150,000
Columnas: 21
Validaciones del modelo OLAP

Conteo de tablas:
dim_date       :      365 filas
dim_customer   :  148,788 filas
dim_vehicle    :        7 filas
dim_location   :   30,564 filas
dim_payment    :        6 filas
dim_status     :       14 filas
fact_bookings  :  150,000 filas

Validación de fact:
Filas origen      : 150,000
Filas fact_bookings: 150,000
✓ La fact conserva el mismo número de registros que el origen

Nulos en llaves de fact:
booking_key     0
date_key        0
customer_key    0
vehicle_key     0
location_key    0
payment_key     0
status_key      0
dtype: int64

Rango de fechas:
Fecha mínima: 2024-01-01 00:00:00
Fecha máxima: 2024-12-30 00:00:00


## hacemos la conexión con aurora

In [ ]:
from sqlalchemy import create_engine, text

# =====================================
# CONFIGURACIÓN AURORA
# =====================================

AURORA_HOST = "aurora-mod4.cluster-ccknelygt3nn.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "CONTRASEÑA"
AURORA_DATABASE = "northwind"

# =====================================
# CREAR ENGINE
# =====================================

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

# =====================================
# PRUEBA DE CONEXIÓN
# =====================================

try:
    with engine.connect() as conn:
        version = conn.execute(
            text("SELECT version();")
        ).scalar()

    print("✅ Conexión exitosa")
    print(version)

except Exception as e:
    print("❌ Error de conexión")
    print(e)

✅ Conexión exitosa
PostgreSQL 17.7 on x86_64-pc-linux-gnu, compiled by x86_64-pc-linux-gnu-gcc (GCC) 10.5.0, 64-bit


In [16]:
SCHEMA = "uber_dwh"

with engine.begin() as conn:
    conn.execute(
        text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
    )

print(f"✅ Schema {SCHEMA} creado correctamente")

✅ Schema uber_dwh creado correctamente


## cargamos la información

In [18]:
tables["dim_vehicle"]

,vehicle_key,vehicle_type
0,1,Auto
1,2,Bike
2,3,Go Mini
3,4,Go Sedan
4,5,Premier Sedan
5,6,Uber XL
6,7,eBike


In [19]:
SCHEMA = "uber_dwh"

tables["dim_vehicle"].to_sql(
    name="dim_vehicle",
    con=engine,
    schema=SCHEMA,
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)

print("✅ dim_vehicle cargada correctamente")

✅ dim_vehicle cargada correctamente


In [20]:
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM uber_dwh.dim_vehicle;")
    ).scalar()

print("Filas en Aurora:", result)

Filas en Aurora: 7


In [21]:
def load_data(tables: dict, engine, schema: str = "uber_dwh") -> None:
    tablas_en_orden = [
        "dim_date",
        "dim_customer",
        "dim_vehicle",
        "dim_location",
        "dim_payment",
        "dim_status",
        "fact_bookings",
    ]

    for nombre in tablas_en_orden:
        tables[nombre].to_sql(
            name=nombre,
            con=engine,
            schema=schema,
            if_exists="replace",
            index=False,
            chunksize=1000,
            method="multi",
        )

        print(f"✅ {nombre:15s} {len(tables[nombre]):>8,} filas cargadas")

    print("\nCarga completa en Aurora PostgreSQL.")

In [22]:
load_data(tables, engine)

✅ dim_date             365 filas cargadas
✅ dim_customer     148,788 filas cargadas
✅ dim_vehicle            7 filas cargadas
✅ dim_location      30,564 filas cargadas
✅ dim_payment            6 filas cargadas
✅ dim_status            14 filas cargadas
✅ fact_bookings    150,000 filas cargadas

Carga completa en Aurora PostgreSQL.


In [23]:
with engine.connect() as conn:
    for tabla in [
        "dim_date",
        "dim_customer",
        "dim_vehicle",
        "dim_location",
        "dim_payment",
        "dim_status",
        "fact_bookings",
    ]:
        total = conn.execute(
            text(f"SELECT COUNT(*) FROM uber_dwh.{tabla};")
        ).scalar()

        print(f"{tabla:15s}: {total:>8,} filas")

dim_date       :      365 filas
dim_customer   :  148,788 filas
dim_vehicle    :        7 filas
dim_location   :   30,564 filas
dim_payment    :        6 filas
dim_status     :       14 filas
fact_bookings  :  150,000 filas


## Hacemos las consultas SQL

In [30]:
def leer_sql(query):
    return pd.read_sql_query(query, con=engine)

¿Cuál fue el desempeño general de Uber durante 2024?

In [32]:
sql_kpis_generales = f"""
WITH kpis AS (
    SELECT
        COUNT(*) AS total_bookings,
        SUM(is_completed) AS completed_bookings,
        SUM(is_cancelled_customer) AS cancelled_by_customer,
        SUM(is_cancelled_driver) AS cancelled_by_driver,
        SUM(is_no_driver_found) AS no_driver_found,
        SUM(is_incomplete) AS incomplete_bookings,
        SUM(COALESCE(booking_value,0)) AS total_revenue
    FROM {SCHEMA}.fact_bookings
)
SELECT
    total_bookings,
    completed_bookings,
    ROUND((completed_bookings * 100.0 / total_bookings)::numeric, 2) AS completion_rate_pct,
    cancelled_by_customer,
    cancelled_by_driver,
    no_driver_found,
    incomplete_bookings,
    ROUND(total_revenue::numeric, 2) AS total_revenue
FROM kpis;
"""

kpis_generales = leer_sql(sql_kpis_generales)
kpis_generales

,total_bookings,completed_bookings,completion_rate_pct,cancelled_by_customer,cancelled_by_driver,no_driver_found,incomplete_bookings,total_revenue
0,150000,93000,62.0,10500,27000,10500,9000,51846183.0


In [44]:
kpi_pie = pd.DataFrame({
    "estatus": [
        "Completados",
        "Cancelados Cliente",
        "Cancelados Conductor",
        "Sin Conductor",
        "Incompletos"
    ],
    "cantidad": [
        kpis_generales["completed_bookings"][0],
        kpis_generales["cancelled_by_customer"][0],
        kpis_generales["cancelled_by_driver"][0],
        kpis_generales["no_driver_found"][0],
        kpis_generales["incomplete_bookings"][0]
    ]
})

fig_kpis = px.pie(
    kpi_pie,
    names="estatus",
    values="cantidad",
    title="Distribución de estados de los viajes"
)

fig_kpis.update_traces(
    textposition="inside",
    textinfo="percent+label"
)

fig_kpis.show()

Vehículos más rentables y desempeño operativo

In [33]:
sql_vehiculos_rentabilidad = f"""
WITH vehicle_metrics AS (
    SELECT
        v.vehicle_type,
        COUNT(*) AS total_bookings,
        SUM(f.is_completed) AS completed_bookings,
        SUM(f.is_cancelled_customer) AS cancelled_by_customer,
        SUM(f.is_cancelled_driver) AS cancelled_by_driver,
        SUM(f.is_no_driver_found) AS no_driver_found,
        SUM(f.is_incomplete) AS incomplete_bookings,
        SUM(COALESCE(f.booking_value, 0)) AS total_revenue,
        AVG(f.booking_value) AS avg_booking_value,
        AVG(f.ride_distance) AS avg_ride_distance,
        AVG(f.avg_vtat) AS avg_vtat,
        AVG(f.avg_ctat) AS avg_ctat
    FROM uber_dwh.fact_bookings f
    INNER JOIN uber_dwh.dim_vehicle v
        ON f.vehicle_key = v.vehicle_key
    GROUP BY
        v.vehicle_type
)
SELECT
    vehicle_type,
    total_bookings,
    completed_bookings,
    ROUND((completed_bookings * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS completion_rate_pct,
    cancelled_by_customer,
    cancelled_by_driver,
    no_driver_found,
    incomplete_bookings,
    ROUND(((cancelled_by_customer + cancelled_by_driver + no_driver_found) * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS operational_failure_rate_pct,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    ROUND(avg_booking_value::numeric, 2) AS avg_booking_value,
    ROUND(avg_ride_distance::numeric, 2) AS avg_ride_distance,
    ROUND(avg_vtat::numeric, 2) AS avg_vtat,
    ROUND(avg_ctat::numeric, 2) AS avg_ctat
FROM vehicle_metrics
ORDER BY
    total_revenue DESC;
"""

vehiculos_rentabilidad = leer_sql(sql_vehiculos_rentabilidad)
vehiculos_rentabilidad

,vehicle_type,total_bookings,completed_bookings,completion_rate_pct,cancelled_by_customer,cancelled_by_driver,no_driver_found,incomplete_bookings,operational_failure_rate_pct,total_revenue,avg_booking_value,avg_ride_distance,avg_vtat,avg_ctat
0,Auto,37419,23155,61.88,2680,6643,2681,2260,32.08,12878422.0,506.73,24.62,8.45,29.14
1,Go Mini,29806,18549,62.23,2097,5330,2015,1815,31.68,10338496.0,507.68,24.61,8.47,29.16
2,Go Sedan,27141,16676,61.44,1832,5031,1960,1642,32.51,9369719.0,511.50,24.61,8.40,29.04
3,Bike,22517,14034,62.33,1575,4077,1503,1328,31.78,7837697.0,510.20,24.65,8.50,29.20
4,Premier Sedan,18111,11252,62.13,1266,3250,1280,1063,32.00,6275332.0,509.57,24.60,8.44,29.22
5,eBike,10557,6551,62.05,723,1907,746,630,31.98,3618485.0,503.90,24.99,8.48,29.18
6,Uber XL,4449,2783,62.55,327,762,315,262,31.56,1528032.0,501.82,24.40,8.58,29.21


In [40]:
fig_vehiculos = px.bar(
    vehiculos_rentabilidad,
    x="vehicle_type",
    y="total_revenue",
    text="total_revenue",
    title="Ingresos totales por tipo de vehículo"
)

fig_vehiculos.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_vehiculos.update_layout(
    xaxis_title="Tipo de vehículo",
    yaxis_title="Ingresos"
)

fig_vehiculos.show()

Evolución mensual de ingresos y crecimiento

In [34]:
sql_evolucion_mensual = f"""
WITH monthly_metrics AS (
    SELECT
        DATE_TRUNC('month', d.fecha)::date AS mes,
        COUNT(*) AS total_bookings,
        SUM(f.is_completed) AS completed_bookings,
        SUM(f.is_cancelled_customer + f.is_cancelled_driver + f.is_no_driver_found) AS operational_failures,
        SUM(COALESCE(f.booking_value, 0)) AS total_revenue
    FROM {SCHEMA}.fact_bookings f
    INNER JOIN {SCHEMA}.dim_date d
        ON f.date_key = d.date_key
    GROUP BY
        DATE_TRUNC('month', d.fecha)::date
),
monthly_with_lag AS (
    SELECT
        mes,
        total_bookings,
        completed_bookings,
        operational_failures,
        total_revenue,
        LAG(total_revenue) OVER (ORDER BY mes) AS previous_month_revenue
    FROM monthly_metrics
)
SELECT
    mes,
    total_bookings,
    completed_bookings,
    ROUND((completed_bookings * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS completion_rate_pct,
    operational_failures,
    ROUND((operational_failures * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS operational_failure_rate_pct,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    ROUND(previous_month_revenue::numeric, 2) AS previous_month_revenue,
    ROUND(((total_revenue - previous_month_revenue) * 100.0 / NULLIF(previous_month_revenue, 0))::numeric, 2) AS revenue_growth_pct
FROM monthly_with_lag
ORDER BY
    mes;
"""

evolucion_mensual = leer_sql(sql_evolucion_mensual)
evolucion_mensual

,mes,total_bookings,completed_bookings,completion_rate_pct,operational_failures,operational_failure_rate_pct,total_revenue,previous_month_revenue,revenue_growth_pct
0,2024-01-01,12861,7951,61.82,4102,31.89,4411069.0,NaN,NaN
1,2024-02-01,11927,7368,61.78,3878,32.51,4085790.0,4411069.0,-7.37
2,2024-03-01,12719,7954,62.54,4038,31.75,4568188.0,4085790.0,11.81
3,2024-04-01,12199,7632,62.56,3842,31.49,4253789.0,4568188.0,-6.88
4,2024-05-01,12778,7905,61.86,4105,32.13,4320679.0,4253789.0,1.57
5,2024-06-01,12440,7757,62.36,3945,31.71,4325660.0,4320679.0,0.12
6,2024-07-01,12897,7926,61.46,4168,32.32,4365923.0,4325660.0,0.93
7,2024-08-01,12636,7780,61.57,4121,32.61,4243509.0,4365923.0,-2.80
8,2024-09-01,12248,7542,61.58,3963,32.36,4191393.0,4243509.0,-1.23
9,2024-10-01,12651,7905,62.49,3985,31.50,4417170.0,4191393.0,5.39


In [41]:
fig_evolucion = px.line(
    evolucion_mensual,
    x="mes",
    y="total_revenue",
    markers=True,
    title="Evolución mensual de ingresos"
)

fig_evolucion.update_layout(
    xaxis_title="Mes",
    yaxis_title="Ingresos"
)

fig_evolucion.show()

¿Qué tipos de vehículo estuvieron en el Top 3 de ingresos en cada mes?

In [35]:
sql_ranking_mensual_vehiculos = f"""
WITH monthly_vehicle_revenue AS (
    SELECT
        DATE_TRUNC('month', d.fecha)::date AS mes,
        v.vehicle_type,
        COUNT(*) AS total_bookings,
        SUM(f.is_completed) AS completed_bookings,
        SUM(COALESCE(f.booking_value, 0)) AS total_revenue
    FROM {SCHEMA}.fact_bookings f
    INNER JOIN {SCHEMA}.dim_date d
        ON f.date_key = d.date_key
    INNER JOIN {SCHEMA}.dim_vehicle v
        ON f.vehicle_key = v.vehicle_key
    GROUP BY
        DATE_TRUNC('month', d.fecha)::date,
        v.vehicle_type
),
ranked_vehicles AS (
    SELECT
        mes,
        vehicle_type,
        total_bookings,
        completed_bookings,
        total_revenue,
        RANK() OVER (
            PARTITION BY mes
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM monthly_vehicle_revenue
)
SELECT
    mes,
    vehicle_type,
    total_bookings,
    completed_bookings,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    revenue_rank
FROM ranked_vehicles
WHERE revenue_rank <= 3
ORDER BY
    mes,
    revenue_rank;
"""

ranking_mensual_vehiculos = leer_sql(sql_ranking_mensual_vehiculos)
ranking_mensual_vehiculos

,mes,vehicle_type,total_bookings,completed_bookings,total_revenue,revenue_rank
0,2024-01-01,Auto,3211,1940,1059174.0,1
1,2024-01-01,Go Mini,2610,1639,891180.0,2
2,2024-01-01,Go Sedan,2239,1375,789039.0,3
3,2024-02-01,Auto,2997,1822,1008107.0,1
4,2024-02-01,Go Mini,2358,1466,837301.0,2
5,2024-02-01,Go Sedan,2193,1339,747544.0,3
6,2024-03-01,Auto,3166,1984,1157233.0,1
7,2024-03-01,Go Mini,2475,1554,882507.0,2
8,2024-03-01,Go Sedan,2273,1437,818031.0,3
9,2024-04-01,Auto,3041,1945,1089526.0,1


In [42]:
fig_ranking = px.bar(
    ranking_mensual_vehiculos,
    x="mes",
    y="total_revenue",
    color="vehicle_type",
    barmode="group",
    title="Top 3 vehículos por ingresos en cada mes"
)

fig_ranking.update_layout(
    xaxis_title="Mes",
    yaxis_title="Ingresos",
    legend_title="Tipo de vehículo"
)

fig_ranking.show()

In [36]:
sql_top_rutas_rentables = f"""
WITH route_metrics AS (
    SELECT
        l.pickup_location,
        l.drop_location,
        COUNT(*) AS total_bookings,
        SUM(f.is_completed) AS completed_bookings,
        SUM(f.is_cancelled_customer + f.is_cancelled_driver + f.is_no_driver_found) AS operational_failures,
        SUM(COALESCE(f.booking_value, 0)) AS total_revenue,
        AVG(f.booking_value) AS avg_booking_value,
        AVG(f.ride_distance) AS avg_ride_distance,
        AVG(f.avg_vtat) AS avg_vtat,
        AVG(f.avg_ctat) AS avg_ctat
    FROM {SCHEMA}.fact_bookings f
    INNER JOIN {SCHEMA}.dim_location l
        ON f.location_key = l.location_key
    GROUP BY
        l.pickup_location,
        l.drop_location
    HAVING COUNT(*) >= 5
)
SELECT
    pickup_location,
    drop_location,
    total_bookings,
    completed_bookings,
    ROUND((completed_bookings * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS completion_rate_pct,
    operational_failures,
    ROUND((operational_failures * 100.0 / NULLIF(total_bookings, 0))::numeric, 2) AS operational_failure_rate_pct,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    ROUND(avg_booking_value::numeric, 2) AS avg_booking_value,
    ROUND(avg_ride_distance::numeric, 2) AS avg_ride_distance,
    ROUND(avg_vtat::numeric, 2) AS avg_vtat,
    ROUND(avg_ctat::numeric, 2) AS avg_ctat
FROM route_metrics
ORDER BY
    total_revenue DESC
LIMIT 15;
"""

top_rutas_rentables = leer_sql(sql_top_rutas_rentables)
top_rutas_rentables

,pickup_location,drop_location,total_bookings,completed_bookings,completion_rate_pct,operational_failures,operational_failure_rate_pct,total_revenue,avg_booking_value,avg_ride_distance,avg_vtat,avg_ctat
0,New Delhi Railway Station,Rajouri Garden,9,4,44.44,3,33.33,9559.0,1593.17,23.80,6.01,27.12
1,Cyber Hub,Gurgaon Railway Station,12,9,75.00,2,16.67,9348.0,934.80,32.52,5.44,26.98
2,Nirman Vihar,Vatika Chowk,9,5,55.56,4,44.44,9284.0,1856.80,29.89,8.50,35.12
3,Ashok Vihar,Basai Dhankot,10,9,90.00,1,10.00,9280.0,1031.11,24.78,8.81,28.28
4,Anand Vihar ISBT,Noida Film City,7,7,100.00,0,0.00,8960.0,1280.00,24.45,9.10,25.01
5,Mayur Vihar,Samaypur Badli,10,7,70.00,1,10.00,8588.0,954.22,17.85,8.15,25.37
6,Model Town,Jahangirpuri,10,6,60.00,2,20.00,8540.0,1067.50,26.12,9.17,26.51
7,Ambience Mall,Akshardham,13,11,84.62,2,15.38,8518.0,774.36,26.94,8.61,26.92
8,Greater Noida,Jor Bagh,8,8,100.00,0,0.00,8252.0,1031.50,32.49,7.08,31.91
9,Noida Extension,Vaishali,9,8,88.89,1,11.11,8202.0,1025.25,16.94,8.54,32.19


In [43]:
top_rutas_rentables["ruta"] = (
    top_rutas_rentables["pickup_location"]
    + " → "
    + top_rutas_rentables["drop_location"]
)

fig_rutas = px.bar(
    top_rutas_rentables.sort_values("total_revenue", ascending=True),
    x="total_revenue",
    y="ruta",
    orientation="h",
    text="total_revenue",
    title="Top 15 rutas más rentables"
)

fig_rutas.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig_rutas.update_layout(
    xaxis_title="Ingresos",
    yaxis_title="Ruta"
)

fig_rutas.show()

Impacto de los tiempos de espera en las cancelaciones

In [47]:
sql_tiempos_cancelaciones = f"""
WITH vehicle_wait_metrics AS (
    SELECT
        v.vehicle_type,
        AVG(f.avg_vtat) AS avg_vehicle_arrival_time,
        AVG(f.avg_ctat) AS avg_customer_wait_time,
        SUM(
            f.is_cancelled_customer +
            f.is_cancelled_driver +
            f.is_no_driver_found +
            f.is_incomplete
        ) AS failed_bookings,
        COUNT(*) AS total_bookings
    FROM {SCHEMA}.fact_bookings f
    INNER JOIN {SCHEMA}.dim_vehicle v
        ON f.vehicle_key = v.vehicle_key
    GROUP BY
        v.vehicle_type
)
SELECT
    vehicle_type,
    ROUND(avg_vehicle_arrival_time::numeric, 2) AS avg_vehicle_arrival_time,
    ROUND(avg_customer_wait_time::numeric, 2) AS avg_customer_wait_time,
    failed_bookings,
    total_bookings,
    ROUND(
        (failed_bookings * 100.0 / NULLIF(total_bookings, 0))::numeric,
        2
    ) AS failure_rate_pct
FROM vehicle_wait_metrics
ORDER BY
    failure_rate_pct DESC;
"""

tiempos_cancelaciones = leer_sql(sql_tiempos_cancelaciones)

tiempos_cancelaciones

,vehicle_type,avg_vehicle_arrival_time,avg_customer_wait_time,failed_bookings,total_bookings,failure_rate_pct
0,Go Sedan,8.40,29.04,10465,27141,38.56
1,Auto,8.45,29.14,14264,37419,38.12
2,eBike,8.48,29.18,4006,10557,37.95
3,Premier Sedan,8.44,29.22,6859,18111,37.87
4,Go Mini,8.47,29.16,11257,29806,37.77
5,Bike,8.50,29.20,8483,22517,37.67
6,Uber XL,8.58,29.21,1666,4449,37.45


Tiempos de espera, fallas e ingresos por tipo de vehículo

In [48]:
sql_impacto_espera_fallas_ingresos = f"""
WITH vehicle_wait_revenue AS (
    SELECT
        dv.vehicle_type,
        COUNT(*) AS total_bookings,

        AVG(f.avg_vtat) AS avg_vtat,
        AVG(f.avg_ctat) AS avg_ctat,

        SUM(
            f.is_cancelled_customer +
            f.is_cancelled_driver +
            f.is_no_driver_found +
            f.is_incomplete
        ) AS failed_bookings,

        AVG(f.booking_value) AS avg_booking_value,
        SUM(COALESCE(f.booking_value, 0)) AS total_revenue

    FROM {SCHEMA}.fact_bookings f
    INNER JOIN {SCHEMA}.dim_vehicle dv
        ON f.vehicle_key = dv.vehicle_key
    GROUP BY
        dv.vehicle_type
)
SELECT
    vehicle_type,
    total_bookings,
    ROUND(avg_vtat::numeric, 2) AS avg_vtat,
    ROUND(avg_ctat::numeric, 2) AS avg_ctat,
    failed_bookings,
    ROUND(
        (100.0 * failed_bookings / NULLIF(total_bookings, 0))::numeric,
        2
    ) AS failure_rate_pct,
    ROUND(avg_booking_value::numeric, 2) AS avg_booking_value,
    ROUND(total_revenue::numeric, 2) AS total_revenue
FROM vehicle_wait_revenue
ORDER BY
    failure_rate_pct DESC;
"""

impacto_espera_fallas_ingresos = leer_sql(sql_impacto_espera_fallas_ingresos)

impacto_espera_fallas_ingresos

,vehicle_type,total_bookings,avg_vtat,avg_ctat,failed_bookings,failure_rate_pct,avg_booking_value,total_revenue
0,Go Sedan,27141,8.40,29.04,10465,38.56,511.50,9369719.0
1,Auto,37419,8.45,29.14,14264,38.12,506.73,12878422.0
2,eBike,10557,8.48,29.18,4006,37.95,503.90,3618485.0
3,Premier Sedan,18111,8.44,29.22,6859,37.87,509.57,6275332.0
4,Go Mini,29806,8.47,29.16,11257,37.77,507.68,10338496.0
5,Bike,22517,8.50,29.20,8483,37.67,510.20,7837697.0
6,Uber XL,4449,8.58,29.21,1666,37.45,501.82,1528032.0
